# 01 — Load + Sanity

Goals:
1. Load Gemma-3-1B-IT on MPS
2. Baseline generation works
3. Capture residual stream at target layer via hook
4. Load one Gemma Scope 2 SAE
5. Encode→decode hidden state, check reconstruction MSE small
6. Note L0 (avg active features) per token

In [8]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

device = 'mps' if torch.backends.mps.is_available() else 'cpu'
print('device:', device)

MODEL_ID = 'google/gemma-3-1b-it'
tok = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map=device,
)
model.eval()
print('n_layers:', model.config.num_hidden_layers, 'd_model:', model.config.hidden_size)

device: mps


Loading weights:   0%|          | 0/340 [00:00<?, ?it/s]

n_layers: 26 d_model: 1152


## Baseline generation

In [9]:
def chat(prompt, max_new=80):
    msgs = [{'role': 'user', 'content': prompt}]
    inputs = tok.apply_chat_template(
        msgs,
        return_tensors='pt',
        return_dict=True,
        add_generation_prompt=True,
    ).to(device)
    in_len = inputs['input_ids'].shape[1]
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_new, do_sample=False)
    return tok.decode(out[0, in_len:], skip_special_tokens=True)

print(chat('Tell me about your morning.'))


Okay, let me tell you about my morning! As a large language model, I don’t *really* experience mornings in the same way a human does. I don’t have a physical body or a need for sleep! But, I can describe what my "morning" looks like – essentially, it’s a continuous process of learning and processing information. 

Here’s a


## Hook residual stream at layer L

Gemma uses `model.model.layers[L]` — capture output of full block (post-residual).

In [10]:
LAYER = 13  # only 13, 17, 22 have resid_post SAEs in gemma-scope-2-1b-it
cache = {}

def cap(_, __, output):
    h = output[0] if isinstance(output, tuple) else output
    cache['resid'] = h.detach()

handle = model.model.layers[LAYER].register_forward_hook(cap)
# direct forward (not generate) — get all token positions, not just last
probe = tok('The quick brown fox jumps over the lazy dog.', return_tensors='pt').to(device)
with torch.no_grad():
    _ = model(**probe)
handle.remove()
print('resid shape:', cache['resid'].shape, cache['resid'].dtype)


resid shape: torch.Size([1, 11, 1152]) torch.bfloat16


## Load Gemma Scope 2 SAE

Try `sae_lens` first. If release ID missing, fall back to direct HF download.

Release IDs to try (check Neuronpedia / HF `google/gemma-scope-2-1b-it-res`):

In [11]:
from sae_lens import SAE

RELEASE = 'gemma-scope-2-1b-it-res'
SAE_ID = f'layer_{LAYER}_width_16k_l0_medium'  # widths: 16k/65k/262k/1m, l0: small/medium/big

sae, cfg, _ = SAE.from_pretrained(release=RELEASE, sae_id=SAE_ID, device=device)
print('loaded:', SAE_ID)
print('d_in:', sae.cfg.d_in, 'd_sae:', sae.cfg.d_sae, 'dtype:', sae.dtype)


config.json:   0%|          | 0.00/247 [00:00<?, ?B/s]

resid_post/layer_13_width_16k_l0_medium/(…):   0%|          | 0.00/151M [00:00<?, ?B/s]

loaded: layer_13_width_16k_l0_medium
d_in: 1152 d_sae: 16384 dtype: torch.float32


/var/folders/k2/qttynbv17pj1sbr20x6pcsc80000gn/T/ipykernel_83276/4151498866.py:6: DeprecationWarning: Unpacking SAE objects is deprecated. SAE.from_pretrained() now returns only the SAE object. Use SAE.from_pretrained_with_cfg_and_sparsity() to get the config dict and sparsity as well.
  sae, cfg, _ = SAE.from_pretrained(release=RELEASE, sae_id=SAE_ID, device=device)


In [12]:
# Manual download fallback (only if sae_lens fails)
# from huggingface_hub import hf_hub_download
# import safetensors.torch as st
# repo = 'google/gemma-scope-2-1b-it'
# path = hf_hub_download(repo_id=repo, filename=f'resid_post/{SAE_ID}/params.safetensors')
# params = st.load_file(path)
# print({k: v.shape for k, v in params.items()})


## Reconstruction sanity

Encode resid → features → decode. MSE should be small relative to resid norm. L0 should match advertised average.

In [13]:
h = cache['resid'].to(torch.float32)  # SAE usually fp32
with torch.no_grad():
    feats = sae.encode(h)
    h_hat = sae.decode(feats)

mse = (h - h_hat).pow(2).mean().item()
norm_sq = h.pow(2).mean().item()
print(f'mse: {mse:.6f}  resid_var: {norm_sq:.6f}  frac_var_unexplained: {mse/norm_sq:.4f}')
print(f'L0 (avg active per token): {(feats > 0).float().sum(-1).mean().item():.1f}')

mse: 479.740570  resid_var: 239894.687500  frac_var_unexplained: 0.0020
L0 (avg active per token): 74.8


## Next

- If recon good (frac unexplained < ~0.2) and L0 sensible: ready for feature discovery → `02_find_features.ipynb`.
- If sae_lens release name wrong, check https://www.neuronpedia.org/gemma-scope-2 for exact IDs.